In [13]:
import pandas as pd
import numpy as np
from libpysal.weights import full2W
from esda.moran import Moran_Local
from pyprojroot import here

In [14]:
RAW_DATA_DIR = here() / "data" / "raw"
INTERIM_DATA_DIR = here() / "data" / "interim"
PROCESSED_DATA_DIR = here() / "data" / "processed"
RESULTS_DATA_DIR = here() / "results"

In [15]:
crime = pd.read_csv(INTERIM_DATA_DIR/"crime_count_2015-2025.csv", encoding='latin-1')
adj = pd.read_csv(INTERIM_DATA_DIR/'adj_combined.csv', encoding='latin-1', index_col=0, dtype=str)

In [16]:
crime["CVEGEO"] = crime["CVEGEO"].astype(str).str.zfill(5)
adj.index = adj.index.astype(str).str.zfill(5)
adj.columns = adj.columns.astype(str).str.zfill(5)

In [17]:
common_ids = sorted(set(adj.index) & set(crime["CVEGEO"]))

In [18]:
adj = adj.loc[common_ids, common_ids]

In [19]:
w = full2W(adj.values.astype(int), ids=common_ids)
w.transform = "r"
print(f"True islands: {w.islands}")

# quadrant code -> label mapping used by esda's Moran_Local.q
QUADRANT_LABELS = {
    1: "HH",  # high-high
    2: "LH",  # low-high
    3: "LL",  # low-low
    4: "HL",  # high-low
}

True islands: []


In [20]:
ALPHA = 0.05
PERMUTATIONS = 999

all_years_results = []
years = sorted(crime["year"].unique())

for year in years:
    yr_data = crime[crime["year"] == year].set_index("CVEGEO").reindex(common_ids)

    if yr_data["crime_count"].isna().any():
        missing = yr_data[yr_data["crime_count"].isna()].index.tolist()
        print(f"Year {year}: missing crime_count for {len(missing)} municipalities — skipping year")
        continue

    y = yr_data["crime_count"].values.astype(float)

    lisa = Moran_Local(y, w, permutations=PERMUTATIONS, seed=42)

    sig = lisa.p_sim < ALPHA
    cluster = np.where(sig, [QUADRANT_LABELS[q] for q in lisa.q], "ns")

    year_df = pd.DataFrame({
        "CVEGEO": common_ids,
        "year": year,
        "crime_count": y,
        "local_I": lisa.Is,
        "quadrant_code": lisa.q,
        "cluster": cluster,          # HH / LH / LL / HL / ns
        "p_sim": lisa.p_sim,
        "significant": sig,
    })

    all_years_results.append(year_df)

    n_sig = sig.sum()
    print(f"Year {year}: {n_sig}/{len(common_ids)} municipalities significant at alpha={ALPHA}")

lisa_results = pd.concat(all_years_results, ignore_index=True)

# merge municipality/state names back in for readability
lisa_results = lisa_results.merge(
    crime[["CVEGEO", "state", "municipality"]].drop_duplicates(),
    on="CVEGEO", how="left"
)

print(lisa_results.head())

Year 2015: 521/2478 municipalities significant at alpha=0.05
Year 2016: 535/2478 municipalities significant at alpha=0.05
Year 2017: 525/2478 municipalities significant at alpha=0.05
Year 2018: 453/2478 municipalities significant at alpha=0.05
Year 2019: 514/2478 municipalities significant at alpha=0.05
Year 2020: 540/2478 municipalities significant at alpha=0.05
Year 2021: 598/2478 municipalities significant at alpha=0.05
Year 2022: 520/2478 municipalities significant at alpha=0.05
Year 2023: 518/2478 municipalities significant at alpha=0.05
Year 2024: 587/2478 municipalities significant at alpha=0.05
Year 2025: 598/2478 municipalities significant at alpha=0.05
  CVEGEO  year  crime_count   local_I  quadrant_code cluster  p_sim  \
0  01001  2015        863.0  0.387858              1      ns  0.223   
1  01002  2015          8.0 -0.082198              2      LH  0.040   
2  01003  2015         17.0 -0.023584              2      LH  0.034   
3  01004  2015         14.0  0.006259        

In [21]:
lisa_results.to_csv(RESULTS_DATA_DIR/"esda/yearly_lisa_results.csv", index=False)

In [22]:
summary = (
    lisa_results
    .groupby(["year", "cluster"])
    .size()
    .unstack(fill_value=0)
)
print(summary)
summary.to_csv(RESULTS_DATA_DIR/"esda/yearly_lisa_cluster_summary.csv")

cluster  HH  HL  LH   LL    ns
year                          
2015     73   1  89  620  2250
2016     71   6  91  636  2229
2017     75   3  88  623  2244
2018     77   2  71  472  2411
2019     75   2  86  566  2304
2020     85   2  97  575  2274
2021     89   1  93  696  2154
2022     90   2  85  578  2278
2023     88   5  82  573  2285
2024     86   4  80  665  2198
2025     99   5  88  667  2174
